In [ ]:
import re
import pandas as pd
from collections import defaultdict

def analyze_ns3_output(log_file_path):
    """
    Parses the ns3 log file to track each hop of every packet.

    Args:
        log_file_path (str): The path to the ns3 log file.

    Returns:
        pandas.DataFrame: A DataFrame where each row is a single hop of a packet.
    """
    packets_info = {}
    packet_hops = defaultdict(list)

    # Regex to parse FlowInfo and LinkTx lines
    flow_info_re = re.compile(r"FlowInfo, .*PktUID: (\d+), OriginalSrc: (\d+), FinalDst: (\d+), Sport: (\d+), FlowTag: (\d+)")
    flow_info_1_re = re.compile(r"FlowInfo-1, PktUID: (\d+)")
    # Updated regex to capture txCompleteTime
    link_tx_re = re.compile(r"LinkTx, Timestamp: (\d+)ns, txCompleteTime: (\d+)ns, .*PktUID: (\d+), LinkSrc: (\d+)")

    with open(log_file_path, 'r') as f:
        for line in f:
            # Match FlowInfo lines to get packet metadata
            flow_match = flow_info_re.search(line)
            if flow_match:
                pkt_uid, src, dst, port, tag = map(int, flow_match.groups())
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': src, 'FinalDst': dst, 'Tag': tag, 'Port': port}
                continue

            # Match FlowInfo-1 lines (for packets without full metadata, like ACKs)
            flow_1_match = flow_info_1_re.search(line)
            if flow_1_match:
                pkt_uid = int(flow_1_match.group(1))
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': None, 'FinalDst': None, 'Tag': None, 'Port': None}
                continue

            # Match LinkTx lines to build the packet path with timestamps
            link_match = link_tx_re.search(line)
            if link_match:
                timestamp, tx_complete_time, pkt_uid, link_src = map(int, link_match.groups())
                current_hops = packet_hops[pkt_uid]
                # Avoid adding duplicate consecutive hops
                if not current_hops or current_hops[-1][0] != link_src:
                    # Store link_src, timestamp, and tx_complete_time for each hop
                    current_hops.append((link_src, timestamp, tx_complete_time))

    # Explode the parsed data into a list of dictionaries, one record per hop
    records = []
    for uid, hops in packet_hops.items():
        info = packets_info.get(uid, {})
        path_nodes = [hop[0] for hop in hops]
        original_src = info.get('OriginalSrc')
        # If OriginalSrc wasn't in the metadata, infer it from the first hop
        if original_src is None and hops:
            original_src = hops[0][0]

        # Iterate through each hop to create a separate record for it
        for i, hop_data in enumerate(hops):
            hop_node, hop_timestamp, tx_complete_time = hop_data
            
            records.append({
                'PktUID': uid,
                'Tag': info.get('Tag'),
                'Port': info.get('Port'),
                'OriginalSrc': original_src,
                'FinalDst': info.get('FinalDst'),
                'HopNumber': i + 1,
                'HopNode': hop_node,
                'HopTimestamp (ns)': hop_timestamp,
                'TxCompleteTime (ns)': tx_complete_time,
                'Path_List': path_nodes
            })

    # Create and return a DataFrame
    df = pd.DataFrame(records)
    if not df.empty:
        # Sort by packet UID and then by hop number to ensure chronological order
        df = df.sort_values(by=['PktUID', 'HopNumber']).reset_index(drop=True)
    return df

# --- Main execution ---
# Path to your ns3 log file
ns3_log_path = "/home/xavid/feina/astra-sim/upc/text_ns3.txt"

# Analyze the log file
packet_hop_df = analyze_ns3_output(ns3_log_path)

# Display the full DataFrame with all individual packet hops.
if not packet_hop_df.empty:
    display(packet_hop_df)
else:
    print("No packet information found in the log file.")

In [ ]:
pd.set_option('display.max_rows', 10)
# packet_hop_df[(packet_hop_df['OriginalSrc'] == 2)]
# packet_hop_df[(packet_hop_df['HopNode'] == 17) & (packet_hop_df['FinalDst'].isin([4,5]))]
packet_hop_df[(packet_hop_df['OriginalSrc'] == 6) & (packet_hop_df['FinalDst'].isin([0]))]
# packet_hop_df[(packet_hop_df['HopNode'].isin([22]))]


In [ ]:
df_interleaved_check = packet_hop_df.copy()

# Exclude rows where FinalDst is NaN (e.g., ACK packets) before analysis
df_interleaved_check.dropna(subset=['FinalDst'], inplace=True)

# List to store records of interleaved transmissions
interleaved_events = []

# Group by the node where the transmission occurs
for hop_node, group in df_interleaved_check.groupby('HopNode'):
    # Sort the entire group by arrival time once
    group = group.sort_values(by='HopTimestamp (ns)').reset_index(drop=True)
    
    # Further group by the specific flow (Source and Port)
    for flow_key, flow_group in group.groupby(['OriginalSrc', 'Port']):
        
        # We need at least two packets in a flow to find an interleaved event
        if len(flow_group) < 2:
            continue
            
        # Iterate through consecutive packets (A and C) in the same flow
        for i in range(len(flow_group) - 1):
            pkt_A = flow_group.iloc[i]
            pkt_C = flow_group.iloc[i+1]
            
            # Get the original indices of A and C from the main group
            start_index = pkt_A.name + 1
            end_index = pkt_C.name
            
            # Check if there are any packets between A and C
            if start_index >= end_index:
                continue

            # Slice the main group to get all potential interleaved packets (B)
            interleaved_candidates = group.iloc[start_index:end_index]
            
            # Find any candidate that is from the same source, a different port, but the same path
            for _, pkt_B in interleaved_candidates.iterrows():
                if (pkt_B['OriginalSrc'] == pkt_A['OriginalSrc']) and \
                   (pkt_B['Port'] != pkt_A['Port']) and \
                   (pkt_B['Path_List'] == pkt_A['Path_List']):
                    
                    # Found an interleaved event. Record A, B, and C.
                    interleaved_events.append(pkt_A)
                    interleaved_events.append(pkt_B)
                    interleaved_events.append(pkt_C)
                    
                    # We only need to find one interleaved packet to confirm the event
                    break 

# Create a DataFrame from the collected interleaved events
if interleaved_events:
    # Create the DataFrame from the list of events
    interleaved_df = pd.DataFrame(interleaved_events)
    
    # Define the columns to use for identifying duplicates (all except the unhashable 'Path_List')
    subset_cols = [col for col in interleaved_df.columns if col != 'Path_List']
    
    # Drop duplicates based on the subset of columns and then sort
    interleaved_df = interleaved_df.drop_duplicates(subset=subset_cols).sort_values(
        by=['HopNode', 'HopTimestamp (ns)', 'PktUID']
    ).reset_index(drop=True)
    
    print("--- Detected Interleaved Transmissions ---")
    print("Showing packets from flow X that were interrupted by a packet from flow Y at the same switch.")
    display(interleaved_df)
else:
    print("No interleaved transmissions were found.")